In [1]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt

# =========================
# CONFIG (SLP)
# =========================
ROOT_DIR = r"C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
PRUNE_TYPE = "prune_layers_ALL"

BATCH_SIZE = 64
PRUNING_LEVELS = [round(x * 0.1, 1) for x in range(11)]  # 0.0, 0.1, ..., 1.0
FILE_PATTERN = "slp_{:.1f}_{}_run_*"

# Directory for raw plots
PLOT_DIR = os.path.join(ROOT_DIR, "SLP_raw_plot")
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 12
})

# =========================
# FUNCTION TO PLOT ALL RUNS
# =========================
def plot_all_runs(metric_col, ylabel, title_text, subtitle_text, filename):
    plt.figure(figsize=(12, 6))

    for run_file in files:
        df = pd.read_csv(run_file, sep=r"\s+")
        df.columns = df.columns.str.strip()
        df[metric_col] = pd.to_numeric(df[metric_col], errors="coerce")
        plt.plot(df["Batch_Number"], df[metric_col], alpha=0.5)

    plt.xlabel("Batch Number")
    plt.ylabel(ylabel)
    plt.title(f"{title_text}\n{subtitle_text}")
    plt.grid(True)
    plt.tight_layout()

    save_path = os.path.join(PLOT_DIR, filename)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()  # Close figure to avoid overlapping plots in loop
    print(f"[INFO] Saved plot: {save_path}")

# =========================
# MAIN LOOP
# =========================
for p in PRUNING_LEVELS:
    folder = os.path.join(ROOT_DIR, f"p-percentage_{p}\\batch_size_{BATCH_SIZE}")
    files = glob.glob(os.path.join(folder, FILE_PATTERN.format(p, BATCH_SIZE)))

    if not files:
        print(f"[WARNING] No files found for pruning {p}, batch size {BATCH_SIZE}")
        continue

    # ---- Cross-Entropy Train ----
    plot_all_runs(
        metric_col="CE_Train",
        ylabel="Cross-Entropy (Train)",
        title_text="SLP - CE Train",
        subtitle_text=f"P%={p}, Batch Size={BATCH_SIZE}",
        filename=f"SLP_CE_Train_P{p}_BS{BATCH_SIZE}.png"
    )

    # ---- Cross-Entropy Test ----
    plot_all_runs(
        metric_col="CE_TEST",
        ylabel="Cross-Entropy (Test)",
        title_text="SLP - CE Test",
        subtitle_text=f"P%={p}, Batch Size={BATCH_SIZE}",
        filename=f"SLP_CE_Test_P{p}_BS{BATCH_SIZE}.png"
    )

    # ---- Accuracy ----
    plot_all_runs(
        metric_col="Accuracy(%)",
        ylabel="Accuracy (%)",
        title_text="SLP - Accuracy",
        subtitle_text=f"P%={p}, Batch Size={BATCH_SIZE}",
        filename=f"SLP_Accuracy_P{p}_BS{BATCH_SIZE}.png"
    )


[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_CE_Train_P0.0_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_CE_Test_P0.0_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_Accuracy_P0.0_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_CE_Train_P0.1_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_CE_Test_P0.1_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_Accuracy_P0.1_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_CE_Train_P0.2_BS64.png
[INFO] Saved plot: C:\Users\S

In [15]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt

# =========================
# CONFIG (SLP)
# =========================
ROOT_DIR = r"C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
PRUNE_TYPE = "prune_layers_ALL"

BATCH_SIZE = 64
PRUNING_LEVELS = [round(x * 0.1, 1) for x in range(11)]  # 0.0, 0.1, ..., 1.0
FILE_PATTERN = "slp_{:.1f}_{}_run_*"

# Directory for raw plots
PLOT_DIR = os.path.join(ROOT_DIR, "SLP_raw_plot")
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 12
})
# =========================
# FUNCTION TO PLOT ALL RUNS
# =========================
def plot_all_runs(metric_col, ylabel, metric_name, pruning_p, batch_size):
    plt.figure(figsize=(12, 6))

    # Collect all values to handle NaNs
    all_values = []

    # Plot all runs
    for run_file in files:
        df = pd.read_csv(run_file, sep=r"\s+")
        df.columns = df.columns.str.strip()
        df[metric_col] = pd.to_numeric(df[metric_col], errors="coerce")
        all_values.extend(df[metric_col].dropna().values)
        plt.plot(df["Batch_Number"], df[metric_col], alpha=0.5)

    # Set y-axis label
    plt.ylabel(ylabel)
    plt.xlabel("Batch Number")
    plt.grid(True)

    # =========================
    # Set y-axis limits robustly
    # =========================
    if metric_name.lower() in ["ce_train", "ce_test"]:
        # Ensure the lower limit is 0 and upper limit is 2.7 even if data has NaNs
        plt.ylim(0, 2.7)
    elif metric_name.lower() == "accuracy":
        plt.ylim(0, 100)

    # =========================
    # Convert pruning % to integer style
    # =========================
    pruning_str = f"P% = {int(pruning_p*100)}%"

    # =========================
    # TITLE HANDLING INSIDE GRID
    # =========================
    if metric_name.lower() == "ce_train":
        vector_type = "Training-Vectors"
        plt.text(0.50, 0.95, "SLP-MNIST", transform=plt.gca().transAxes, ha='center', va='top')
        plt.text(
            0.50, 0.90,
            f"({vector_type}, {pruning_str}, BS={batch_size})",
            ha='center', fontsize=16, transform=plt.gca().transAxes, va='top'
        )
    elif metric_name.lower() == "ce_test":
        vector_type = "Test-Vectors"
        plt.text(0.50, 0.95, "SLP-MNIST", transform=plt.gca().transAxes, ha='center', va='top')
        plt.text(
            0.50, 0.90,
            f"({vector_type}, {pruning_str}, BS={batch_size})",
            ha='center', fontsize=16, transform=plt.gca().transAxes, va='top'
        )
    elif metric_name.lower() == "accuracy":
        vector_type = "Test-Vectors"
        plt.text(0.50, 0.55, "SLP-MNIST", transform=plt.gca().transAxes, ha='center', va='top')
        plt.text(
            0.50, 0.50,
            f"({vector_type}, {pruning_str}, BS={batch_size})",
            ha='center', fontsize=16, transform=plt.gca().transAxes, va='top'
        )

    plt.tight_layout()

    # Save plot
    filename = f"SLP_{metric_name}_P{int(pruning_p*100)}_BS{batch_size}.png"
    save_path = os.path.join(PLOT_DIR, filename)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[INFO] Saved plot: {save_path}")

# =========================
# MAIN LOOP
# =========================
for p in PRUNING_LEVELS:
    folder = os.path.join(ROOT_DIR, f"p-percentage_{p}\\batch_size_{BATCH_SIZE}")
    files = glob.glob(os.path.join(folder, FILE_PATTERN.format(p, BATCH_SIZE)))

    if not files:
        print(f"[WARNING] No files found for pruning {p}, batch size {BATCH_SIZE}")
        continue

    # ---- Cross-Entropy Train ----
    plot_all_runs(
        metric_col="CE_Train",
        ylabel="Cross-Entropy",
        metric_name="CE_Train",
        pruning_p=p,
        batch_size=BATCH_SIZE
    )

    # ---- Cross-Entropy Test ----
    plot_all_runs(
        metric_col="CE_TEST",
        ylabel="Cross-Entropy",
        metric_name="CE_Test",
        pruning_p=p,
        batch_size=BATCH_SIZE
    )

    # ---- Accuracy ----
    plot_all_runs(
        metric_col="Accuracy(%)",
        ylabel="Accuracy (%)",
        metric_name="Accuracy",
        pruning_p=p,
        batch_size=BATCH_SIZE
    )


[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_CE_Train_P0_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_CE_Test_P0_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_Accuracy_P0_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_CE_Train_P10_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_CE_Test_P10_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_Accuracy_P10_BS64.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot\SLP_CE_Train_P20_BS64.png
[INFO] Saved plot: C:\Users\Student\Des

In [24]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt

# =========================
# CONFIG (SLP)
# =========================
ROOT_DIR = r"C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
PRUNE_TYPE = "prune_layers_ALL"

BATCH_SIZE = 60000
PRUNING_LEVELS = [round(x * 0.1, 1) for x in range(11)]  # 0.0, 0.1, ..., 1.0
FILE_PATTERN = "slp_{:.1f}_{}_run_*"

# Directory for raw plots
PLOT_DIR = os.path.join(ROOT_DIR, "SLP_raw_plot_60000")
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# STYLE
# =========================
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 12
})

# =========================
# FUNCTION TO PLOT ALL RUNS
# =========================
def plot_all_runs(metric_col, ylabel, metric_name, pruning_p, batch_size, files):
    plt.figure(figsize=(12, 6))

    # Plot all runs
    for run_file in files:
        df = pd.read_csv(run_file, sep=r"\s+")
        df.columns = df.columns.str.strip()
        df[metric_col] = pd.to_numeric(df[metric_col], errors="coerce")
        plt.plot(df["Batch_Number"], df[metric_col], alpha=0.5)

    # =========================
    # Set y-axis limits BEFORE adding text
    # =========================
    if metric_name.lower() in ["ce_train", "ce_test"]:
        plt.ylim(0, 2.7)
    elif metric_name.lower() == "accuracy":
        plt.ylim(0, 100)

    # Labels
    plt.ylabel(ylabel)
    plt.xlabel("Batch Number")
    plt.grid(True)

    # Pruning string
    pruning_str = f"P% = {int(pruning_p*100)}%"

    # =========================
    # Fixed TITLE POSITIONS
    # =========================
    if metric_name.lower() == "ce_train":
        vector_type = "Training-Vectors"
        # Place titles just below top of the axes (e.g., 0.95 and 0.91)
        plt.text(0.5, 0.97, "SLP-MNIST", transform=plt.gca().transAxes, ha='center', va='top')
        plt.text(0.5, 0.93, f"({vector_type}, {pruning_str}, BS={batch_size})",
                 ha='center', fontsize=16, transform=plt.gca().transAxes, va='top')
    
    elif metric_name.lower() == "ce_test":
        vector_type = "Test-Vectors"
        # Fixed distance from top
        plt.text(0.5, 0.97, "SLP-MNIST", transform=plt.gca().transAxes, ha='center', va='top')
        plt.text(0.5, 0.93, f"({vector_type}, {pruning_str}, BS={batch_size})",
                 ha='center', fontsize=16, transform=plt.gca().transAxes, va='top')


    elif metric_name.lower() == "accuracy":
        vector_type = "Test-Vectors"
        plt.text(0.50, 0.55, "SLP-MNIST", transform=plt.gca().transAxes, ha='center', va='top')
        plt.text(0.50, 0.50, f"({vector_type}, {pruning_str}, BS={batch_size})",
                 ha='center', fontsize=16, transform=plt.gca().transAxes, va='top')

    plt.tight_layout()

    # Save plot
    filename = f"SLP_{metric_name}_P{int(pruning_p*100)}_BS{batch_size}.png"
    save_path = os.path.join(PLOT_DIR, filename)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[INFO] Saved plot: {save_path}")


# =========================
# MAIN LOOP
# =========================
for p in PRUNING_LEVELS:
    folder = os.path.join(ROOT_DIR, f"p-percentage_{p}\\batch_size_{BATCH_SIZE}")
    files = glob.glob(os.path.join(folder, FILE_PATTERN.format(p, BATCH_SIZE)))

    if not files:
        print(f"[WARNING] No files found for pruning {p}, batch size {BATCH_SIZE}")
        continue

    # ---- Cross-Entropy Train ----
    plot_all_runs(
        metric_col="CE_Train",
        ylabel="Cross-Entropy",
        metric_name="ce_train",
        pruning_p=p,
        batch_size=BATCH_SIZE,
        files=files
    )

    # ---- Cross-Entropy Test ----
    plot_all_runs(
        metric_col="CE_TEST",
        ylabel="Cross-Entropy",
        metric_name="ce_test",
        pruning_p=p,
        batch_size=BATCH_SIZE,
        files=files
    )

    # ---- Accuracy ----
    plot_all_runs(
        metric_col="Accuracy(%)",
        ylabel="Accuracy (%)",
        metric_name="accuracy",
        pruning_p=p,
        batch_size=BATCH_SIZE,
        files=files
    )


[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot_60000\SLP_ce_train_P0_BS60000.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot_60000\SLP_ce_test_P0_BS60000.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot_60000\SLP_accuracy_P0_BS60000.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot_60000\SLP_ce_train_P10_BS60000.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot_60000\SLP_ce_test_P10_BS60000.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot_60000\SLP_accuracy_P10_BS60000.png
[INFO] Saved plot: C:\Users\Student\Desktop\Neural_research\physlab\SLP\SLP-MNIST\prune_layers_ALL\SLP_raw_plot_60000\SLP_c